<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 4 — Intégration et Contrôles Inter-sources

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif

Construire le **dataset analytique final** `dataset_complet.csv` en fusionnant
les trois bases nettoyées des Phases 1, 2 et 3.

Ce dataset constitue la **base unique** pour toutes les analyses ultérieures (Phases 5 & 6).

**Stratégie de jointure :**
```
dossier_nettoye (98 935 lignes — unité d'analyse)
    ├── LEFT JOIN temps_agrégé       (sur Numero_dossier_ID = Numero.dossier)
    └── LEFT JOIN ressources         (sur Matricule.de.traitement = Matricule)
```

Le LEFT JOIN depuis `dossier` garantit que **toutes les 98 935 lignes** sont conservées.


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine (présence de .git ou requirements.txt)."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))
from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
# --- Chargement des trois bases nettoyées
dossier    = pd.read_csv('data/dossier_nettoye.csv',     encoding='utf-8')
ressources = pd.read_csv('data/ressources_nettoyees.csv', encoding='utf-8')
temps      = pd.read_csv('data/temps_nettoye.csv',        encoding='utf-8')

print(f" dossier    : {dossier.shape[0]:,} lignes × {dossier.shape[1]} colonnes")
print(f" ressources : {ressources.shape[0]:,} lignes × {ressources.shape[1]} colonnes")
print(f" temps      : {temps.shape[0]:,} lignes × {temps.shape[1]} colonnes")
print()
print("Clés de jointure :")
print(f"  dossier.Numero_dossier_ID    ↔ temps.Numero.dossier")
print(f"  dossier.Matricule.de.traitement ↔ ressources.Matricule")

---
## Section 1 — D1 : Agrégation de `temps` par dossier

### Objectif
Synthétiser toutes les interventions d'un dossier en une seule ligne d'indicateurs.

### Décision : LEFT JOIN — conservation des dossiers sans intervention
> Les 2 180 dossiers sans intervention dans `temps` obtiendront des NaN pour ces variables.  
> NaN ≠ durée nulle : l'absence d'enregistrement est une information distincte de zéro.

In [ ]:
# --- Agrégation de temps par dossier
temps_agrege = (
    temps.groupby('Numero.dossier')
    .agg(
        nb_interventions          = ('Matricule', 'count'),
        nb_agents_distincts       = ('Matricule', 'nunique'),
        duree_totale_min          = ('duree.corrigee', 'sum'),
        duree_moyenne_min         = ('duree.corrigee', 'mean'),
        date_premiere_intervention = ('Date.debut.traitement', 'min'),
        date_derniere_intervention = ('Date.debut.traitement', 'max'),
    )
    .reset_index()
    .rename(columns={'Numero.dossier': 'Numero_dossier_ID'})
)

print(f" Agrégat temps : {len(temps_agrege):,} dossiers uniques")
print()
print("Statistiques de l'agrégat :")
print(temps_agrege[['nb_interventions','nb_agents_distincts',
                     'duree_totale_min','duree_moyenne_min']].describe().round(2))
print()
print("Aperçu :")
temps_agrege.head(5).style_duplicates()

In [ ]:
# Vérification : nb_interventions vs nb_agents_distincts
dossiers_multi_agents = (temps_agrege['nb_agents_distincts'] > 1).sum()
dossiers_un_agent     = (temps_agrege['nb_agents_distincts'] == 1).sum()
print(f"Dossiers traités par 1 seul agent    : {dossiers_un_agent:,}")
print(f"Dossiers traités par plusieurs agents : {dossiers_multi_agents:,}")
print()
print("Distribution du nb d'agents par dossier :")
print(temps_agrege['nb_agents_distincts'].value_counts().sort_index().head(10))

---
## Section 2 — D2 : Extraction des caractéristiques de l'agent depuis `ressources`

### Stratégie
Pour chaque dossier, récupérer les caractéristiques de l'agent `Matricule.de.traitement`
à la date la plus proche de `date.ouverture`.

**Algorithme :**
1. Pour chaque dossier, récupérer son `Matricule.de.traitement` et sa `date.ouverture`
2. Chercher dans `ressources` la ligne pour ce matricule avec `Date.presence <= date.ouverture`
3. Prendre la ligne avec la `Date.presence` la plus récente (la plus proche de la date d'ouverture)
4. Récupérer les colonnes souhaitées